In [1]:
# ============================================================
# Autosave helpers for Quad4FHE plaintext experiments
# - duplicates notebook stdout/stderr to a .txt log
# - writes structured JSON after each quad4fhe.replace(...) run
# - writes a compact CSV summary
# ============================================================
from __future__ import annotations

import contextlib
import json
import math
import re
import sys
import traceback
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, Optional


_AUTOSAVE_PAYLOAD: Optional[Dict[str, Any]] = None
_AUTOSAVE_RUNS = []
_AUTOSAVE_SUMMARY_ROWS = []


class _AutosaveTeeStream:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

    def isatty(self):
        return any(getattr(stream, "isatty", lambda: False)() for stream in self.streams)


@contextlib.contextmanager
def tee_stdout_stderr(log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    old_stdout, old_stderr = sys.stdout, sys.stderr
    with log_path.open("w", encoding="utf-8", buffering=1) as fh:
        sys.stdout = _AutosaveTeeStream(old_stdout, fh)
        sys.stderr = _AutosaveTeeStream(old_stderr, fh)
        try:
            print(f"[autosave] stdout/stderr log -> {log_path}")
            yield log_path
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def _to_jsonable(obj: Any) -> Any:
    try:
        import numpy as _np
    except Exception:
        _np = None
    try:
        import pandas as _pd
    except Exception:
        _pd = None
    try:
        import torch as _torch
    except Exception:
        _torch = None

    if obj is None or isinstance(obj, (str, bool, int)):
        return obj
    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None
    if _np is not None:
        if isinstance(obj, _np.integer):
            return int(obj)
        if isinstance(obj, _np.floating):
            value = float(obj)
            return value if math.isfinite(value) else None
        if isinstance(obj, _np.bool_):
            return bool(obj)
        if isinstance(obj, _np.ndarray):
            return _to_jsonable(obj.tolist())
    if _torch is not None and hasattr(_torch, "is_tensor") and _torch.is_tensor(obj):
        return _to_jsonable(obj.detach().cpu().numpy())
    if isinstance(obj, Path):
        return str(obj)
    if _pd is not None:
        if isinstance(obj, _pd.DataFrame):
            return _to_jsonable(obj.to_dict(orient="records"))
        if isinstance(obj, _pd.Series):
            return _to_jsonable(obj.to_dict())
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_to_jsonable(v) for v in obj]
    return str(obj)


def _save_json(obj: Dict[str, Any], path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(_to_jsonable(obj), fh, indent=2, ensure_ascii=False, allow_nan=False)
    print(f"[autosave] JSON -> {path}")
    return path


def _save_csv(rows: Iterable[Dict[str, Any]], path) -> Path:
    import pandas as _pd
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    _pd.DataFrame(list(rows)).to_csv(path, index=False)
    print(f"[autosave] CSV -> {path}")
    return path


def _dataframe_records(df: Any) -> list:
    if df is None:
        return []
    try:
        return _to_jsonable(df.to_dict(orient="records"))
    except Exception:
        return []


def _dataframe_test_by_method(df: Any) -> Dict[str, Dict[str, Any]]:
    out: Dict[str, Dict[str, Any]] = {}
    if df is None:
        return out
    try:
        sub = df[df["Split"] == "test"]
        for _, row in sub.iterrows():
            method = str(row.get("Method"))
            metrics = {str(k): _to_jsonable(v) for k, v in row.to_dict().items()
                       if k not in ("Method", "Split")}
            out[method] = metrics
    except Exception:
        return out
    return out


def _metric_from_table(table: Dict[str, Dict[str, Any]], method: str, *keys: str) -> Any:
    row = table.get(method, {})
    for key in keys:
        if key in row:
            return row[key]
    return None


def _quad_report_diagnostics(result: Any, split: str, n_expected: Optional[int] = None) -> Dict[str, Any]:
    if split in ("fit", "calib", "calibration"):
        attr_candidates = ["fit_diagnostics", "calib_diagnostics", "calibration_diagnostics"]
        split_label = "fit"
    else:
        attr_candidates = ["test_diagnostics"]
        split_label = "test"

    diag = None
    for attr in attr_candidates:
        value = getattr(result, attr, None)
        if value is not None:
            diag = dict(value)
            break

    if diag is None:
        diag = {}
        df = getattr(result, "report_vs_pseudo", None)
        if df is not None:
            try:
                row = df[(df["Method"] == "Quad4FHE") & (df["Split"] == split_label)]
                if len(row) > 0:
                    diag["agreement"] = float(row.iloc[0]["Agreement"])
            except Exception:
                pass

    n_value = diag.get("n", diag.get("calib_n", diag.get("n_samples", n_expected)))
    agreement = diag.get("agreement", diag.get("calib_agreement", diag.get("fit_agreement")))
    mismatch = diag.get("mismatch_count", diag.get("calib_mismatch_count", diag.get("fit_mismatch_count")))
    if mismatch is None and agreement is not None and n_value is not None:
        mismatch = int(round((1.0 - float(agreement)) * int(n_value)))

    exact = diag.get("exact_preserved", diag.get("exact_preserved_on_calib", diag.get("fit_exact_preserved")))
    if exact is None and mismatch is not None:
        exact = bool(int(mismatch) == 0)

    return {
        "split": split_label,
        "n": _to_jsonable(n_value),
        "agreement": _to_jsonable(agreement),
        "mismatch_count": _to_jsonable(mismatch),
        "exact_preserved": _to_jsonable(exact),
    }


def _quad_solver_diagnostics(result: Any) -> Dict[str, Any]:
    return _to_jsonable(dict(getattr(result, "solver_diagnostics", None) or {}))


def _build_quad4fhe_run_record(
    *,
    result: Any,
    key: str,
    hidden_dim: Optional[int],
    fit_n: Optional[int],
    test_n: Optional[int],
    pool_fraction: Optional[float] = None,
    rep_fit_size: Optional[int] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    fit_diag = _quad_report_diagnostics(result, "fit", n_expected=fit_n)
    test_diag = _quad_report_diagnostics(result, "test", n_expected=test_n)
    solver_diag = _quad_solver_diagnostics(result)

    report_truth_test = _dataframe_test_by_method(getattr(result, "report_vs_truth", None))
    report_pseudo_test = _dataframe_test_by_method(getattr(result, "report_vs_pseudo", None))

    calib_agreement = fit_diag.get("agreement")
    test_agreement = test_diag.get("agreement")
    gap = None
    if calib_agreement is not None and test_agreement is not None:
        gap = float(calib_agreement) - float(test_agreement)

    constraint_version = getattr(result, "constraint_version", None)
    if constraint_version is None:
        constraint_version = solver_diag.get("constraint_version")

    common_solver_keys = [
        "num_pairwise_constraints",
        "min_pairwise_margin",
        "normalized_min_pairwise_margin",
        "slack_positive_count",
        "sum_slack",
        "mean_slack",
        "max_slack",
        "pairwise_slack_positive_count",
        "pairwise_sum_slack",
        "pairwise_mean_slack",
        "pairwise_max_slack",
        "selected_C",
        "soft_C_grid",
        "soft_trace",
        "selected_mu",
        "mu_grid",
        "mu_p",
        "mu_n",
    ]

    quad = {
        "alpha": getattr(result, "alpha", None),
        "beta": getattr(result, "beta", None),
        "eta": getattr(result, "eta", None),
        "threshold": getattr(result, "threshold", None),
        "zero_threshold_realized": getattr(result, "zero_threshold_realized", None),
        "method_used": getattr(result, "method_used", None),
        "hard_feasible": getattr(result, "feasible", None),
        "empirical_margin": getattr(result, "empirical_margin", None),
        "normalized_margin": getattr(result, "normalized_margin", None),
        "quant_decimals": getattr(result, "quant_decimals", None),
        "constraint_version": constraint_version,
        "he_artifact_dir": str(getattr(result, "he_export_dir", None)) if getattr(result, "he_export_dir", None) else None,
        "calib_n": fit_diag.get("n"),
        "calib_agreement": calib_agreement,
        "calib_mismatch_count": fit_diag.get("mismatch_count"),
        "exact_preserved_on_calib": fit_diag.get("exact_preserved"),
        "test_n": test_diag.get("n"),
        "test_agreement": test_agreement,
        "test_mismatch_count": test_diag.get("mismatch_count"),
        "calib_test_agreement_gap": gap,
        "test_top1_acc": _metric_from_table(report_truth_test, "Quad4FHE", "ACC", "Top1", "Top-1"),
        "test_top5_acc": _metric_from_table(report_truth_test, "Quad4FHE", "Top5", "Top-5"),
        "test_macro_f1": _metric_from_table(report_truth_test, "Quad4FHE", "MacroF1", "F1"),
        "solver_diagnostics": solver_diag,
    }
    for k in common_solver_keys:
        quad[k] = solver_diag.get(k)

    original = {
        "test_top1_acc": _metric_from_table(report_truth_test, "Original", "ACC", "Top1", "Top-1"),
        "test_top5_acc": _metric_from_table(report_truth_test, "Original", "Top5", "Top-5"),
    }

    record = {
        "key": key,
        "hidden_dim": hidden_dim,
        "pool_fraction": pool_fraction,
        "rep_fit_size": rep_fit_size,
        "original": original,
        "quad4fhe": quad,
        "report_vs_ground_truth_test": report_truth_test,
        "report_vs_pseudo_labels_test": report_pseudo_test,
        "report_vs_ground_truth_records": _dataframe_records(getattr(result, "report_vs_truth", None)),
        "report_vs_pseudo_labels_records": _dataframe_records(getattr(result, "report_vs_pseudo", None)),
    }
    if extra:
        record.update(_to_jsonable(extra))
    return _to_jsonable(record)


def _collect_autosave_config() -> Dict[str, Any]:
    keys = [
        "SEED", "HIDDEN_DIMS", "HIDDEN_DIM", "EPOCHS", "LR", "BATCH_SIZE", "WEIGHT_DECAY",
        "TRAIN_SIZE", "VAL_SIZE", "TEST_SIZE", "TRAIN_NET_SIZE", "VAL_NET_SIZE",
        "MLP_VAL_FRACTION", "POOL_SIZES_TO_COMPARE", "REPLACEMENT_SELECTION_FRACTION",
        "MIN_POOL_SAMPLES", "MIN_SAMPLES_PER_CLASS", "MIN_PRESENT_CLASSES",
        "NUM_CLASSES", "FIT_SUBSAMPLE_PER_CLASS", "PCA_COMPONENTS", "SVD_COMPONENTS",
        "TFIDF_MAX_FEATURES", "TFIDF_NGRAM_RANGE", "BASELINES", "FLIP_LABELS",
        "USE_HF_DATASETS", "SHUTTLE_SOURCE",
    ]
    g = globals()
    return {k: _to_jsonable(g[k]) for k in keys if k in g}


def _make_autosave_payload() -> Dict[str, Any]:
    g = globals()
    return {
        "dataset": g.get("DATASET_NAME"),
        "experiment": g.get("EXPERIMENT_NAME"),
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "source_notebook": g.get("SOURCE_NOTEBOOK", None),
        "log_file": str(g.get("LOG_PATH")),
        "seed": g.get("SEED", None),
        "dataset_info": {},
        "config": _collect_autosave_config(),
        "runs": [],
    }


def _ensure_autosave_payload() -> Dict[str, Any]:
    global _AUTOSAVE_PAYLOAD
    if _AUTOSAVE_PAYLOAD is None:
        _AUTOSAVE_PAYLOAD = _make_autosave_payload()
    return _AUTOSAVE_PAYLOAD


def _infer_dataset_info_from_runs(payload: Dict[str, Any]) -> Dict[str, Any]:
    info = dict(payload.get("dataset_info", {}) or {})
    runs = payload.get("runs", [])
    if runs:
        q0 = runs[0].get("quad4fhe", {})
        info.setdefault("calib_n_first", q0.get("calib_n"))
        info.setdefault("test_n", q0.get("test_n"))
        if "NUM_CLASSES" in globals():
            info.setdefault("num_classes", globals().get("NUM_CLASSES"))
        if "PCA_COMPONENTS" in globals():
            info.setdefault("input_dim", globals().get("PCA_COMPONENTS"))
        elif "SVD_COMPONENTS" in globals():
            info.setdefault("input_dim", globals().get("SVD_COMPONENTS"))
    return _to_jsonable(info)


def _summary_row_from_record(record: Dict[str, Any]) -> Dict[str, Any]:
    q = record.get("quad4fhe", {})
    return {
        "key": record.get("key"),
        "hidden_dim": record.get("hidden_dim"),
        "pool_fraction": record.get("pool_fraction"),
        "rep_fit_size": record.get("rep_fit_size"),
        "status": record.get("status", "ok"),
        "method_used": q.get("method_used"),
        "hard_feasible": q.get("hard_feasible"),
        "alpha": q.get("alpha"),
        "beta": q.get("beta"),
        "eta": q.get("eta"),
        "empirical_margin": q.get("empirical_margin"),
        "normalized_margin": q.get("normalized_margin"),
        "quant_decimals": q.get("quant_decimals"),
        "calib_n": q.get("calib_n"),
        "calib_agreement": q.get("calib_agreement"),
        "calib_mismatch_count": q.get("calib_mismatch_count"),
        "exact_preserved_on_calib": q.get("exact_preserved_on_calib"),
        "test_agreement": q.get("test_agreement"),
        "test_mismatch_count": q.get("test_mismatch_count"),
        "calib_test_agreement_gap": q.get("calib_test_agreement_gap"),
        "num_pairwise_constraints": q.get("num_pairwise_constraints"),
        "min_pairwise_margin": q.get("min_pairwise_margin"),
        "slack_positive_count": q.get("slack_positive_count"),
        "sum_slack": q.get("sum_slack"),
        "max_slack": q.get("max_slack"),
        "selected_C": q.get("selected_C"),
        "constraint_version": q.get("constraint_version"),
        "test_top1_acc": q.get("test_top1_acc"),
        "test_top5_acc": q.get("test_top5_acc"),
        "he_artifact_dir": q.get("he_artifact_dir"),
        "skip_reason": record.get("skip_reason"),
    }


def _autosave_write_current_payload() -> None:
    payload = _ensure_autosave_payload()
    payload["dataset_info"] = _infer_dataset_info_from_runs(payload)
    payload["config"] = _collect_autosave_config()
    payload["summary"] = _AUTOSAVE_SUMMARY_ROWS
    if "JSON_PATH" in globals():
        _save_json(payload, globals()["JSON_PATH"])
    if "SUMMARY_CSV_PATH" in globals() and _AUTOSAVE_SUMMARY_ROWS:
        _save_csv(_AUTOSAVE_SUMMARY_ROWS, globals()["SUMMARY_CSV_PATH"])
    _autosave_write_combined_dataset_json()


def _autosave_write_combined_dataset_json() -> None:
    g = globals()
    dataset = g.get("DATASET_NAME")
    if not dataset:
        return
    root = Path("..") / "results" / dataset
    combined = {"dataset": dataset, "created_at": datetime.now().isoformat(timespec="seconds")}
    found = False
    for exp in ("fulltrain", "smallpool"):
        p = root / exp / f"{dataset}_{exp}_results.json"
        if not p.exists():
            continue
        try:
            with p.open("r", encoding="utf-8") as fh:
                block = json.load(fh)
            combined[exp] = {
                "source_json": str(p),
                "source_notebook": block.get("source_notebook"),
                "log_file": block.get("log_file"),
                "dataset_info": block.get("dataset_info", {}),
                "config": block.get("config", {}),
                "runs": block.get("runs", []),
                "summary": block.get("summary", []),
            }
            found = True
        except Exception as exc:
            combined[f"{exp}_read_error"] = str(exc)
    if found:
        _save_json(combined, root / f"{dataset}_results.json")


def _autosave_record_result(
    *,
    result: Any,
    key: str,
    hidden_dim: Optional[int],
    fit_n: Optional[int],
    test_n: Optional[int],
    pool_fraction: Optional[float] = None,
    rep_fit_size: Optional[int] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    payload = _ensure_autosave_payload()
    record = _build_quad4fhe_run_record(
        result=result,
        key=key,
        hidden_dim=hidden_dim,
        fit_n=fit_n,
        test_n=test_n,
        pool_fraction=pool_fraction,
        rep_fit_size=rep_fit_size,
        extra=extra,
    )
    payload["runs"].append(record)
    _AUTOSAVE_SUMMARY_ROWS.append(_summary_row_from_record(record))
    _autosave_write_current_payload()
    return record


def _autosave_record_skipped(
    *,
    key: str,
    hidden_dim: Optional[int],
    pool_fraction: Optional[float],
    reason: str,
    rep_fit_size: Optional[int] = None,
) -> Dict[str, Any]:
    payload = _ensure_autosave_payload()
    record = {
        "key": key,
        "hidden_dim": hidden_dim,
        "pool_fraction": pool_fraction,
        "rep_fit_size": rep_fit_size,
        "status": "skipped",
        "skip_reason": reason,
        "quad4fhe": {"status": "skipped", "skip_reason": reason},
    }
    payload["runs"].append(_to_jsonable(record))
    _AUTOSAVE_SUMMARY_ROWS.append(_summary_row_from_record(record))
    _autosave_write_current_payload()
    return record


def run_notebook_main_with_autosave(main_fn, log_path):
    global _AUTOSAVE_PAYLOAD, _AUTOSAVE_RUNS, _AUTOSAVE_SUMMARY_ROWS
    _AUTOSAVE_PAYLOAD = None
    _AUTOSAVE_RUNS = []
    _AUTOSAVE_SUMMARY_ROWS = []
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    try:
        with tee_stdout_stderr(log_path):
            print(f"[autosave] dataset={globals().get('DATASET_NAME')} experiment={globals().get('EXPERIMENT_NAME')}")
            print(f"[autosave] output_dir={globals().get('OUTPUT_DIR')}")
            main_fn()
            _autosave_write_current_payload()
    except Exception:
        # Write partial JSON before re-raising, useful for debugging interrupted runs.
        err = traceback.format_exc()
        payload = _ensure_autosave_payload()
        payload["error"] = err
        if "JSON_PATH" in globals():
            _save_json(payload, globals()["JSON_PATH"])
        raise


In [2]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import quad4fhe


# ============================================================
# Autosave configuration
# ============================================================
from pathlib import Path
DATASET_NAME = "CIFAR10"
EXPERIMENT_NAME = "smallpool"
SOURCE_NOTEBOOK = "CIFAR10_smallpool_autosave.ipynb"
OUTPUT_DIR = Path("..") / "results" / DATASET_NAME / EXPERIMENT_NAME
LOG_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_result.txt"
JSON_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_results.json"
SUMMARY_CSV_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_summary.csv"

from pathlib import Path


# ============================================================
# Configuration
# ============================================================
SEED = 2026
HIDDEN_DIM = 256
EPOCHS = 200
LR = 1e-3
BATCH_SIZE = 512

EARLY_STOP_PATIENCE = 10
EARLY_STOP_MIN_DELTA = 1e-4
WEIGHT_DECAY = 1e-5

VAL_NET_FRAC   = 0.10     # 10% of TOTAL (60k) -> 6k, from train.csv
POOL_MAX_FRAC  = 0.20     # 20% of TOTAL (60k) -> 12k, from train.csv (disjoint from val_net)
REPLACEMENT_SELECTION_FRACTION = 0.5

POOL_SIZES_TO_COMPARE = [0.01, 0.05, 0.10, 0.20]

# CIFAR-10 is perfectly balanced — standard guards work fine.
MIN_POOL_SAMPLES = 50
MIN_SAMPLES_PER_CLASS = 2

# --- CIFAR-10 data ---
CIFAR_DATA_ROOT = "./data"

# --- Feature extraction ---
PCA_COMPONENTS = 256

NUM_CLASSES = 10

BASELINES = ["square", "ls_poly_2", "ls_poly_3", "ls_poly_5", "ls_poly_7",
             "remez_2", "remez_3", "remez_5", "remez_7",
             "ola", "precise_a11"]


# ============================================================
# Model + EarlyStopping
# ============================================================
class MulticlassMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn  = nn.BatchNorm1d(hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        return self.fc2(self.act(self.bn(self.fc1(x))))


class EarlyStopping:
    def __init__(self, patience=EARLY_STOP_PATIENCE, min_delta=EARLY_STOP_MIN_DELTA):
        self.patience, self.min_delta = patience, min_delta
        self.best_loss = float("inf"); self.counter = 0
        self.best_epoch = 0; self.best_state = None

    def step(self, val_loss, epoch, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss, self.counter, self.best_epoch = val_loss, 0, epoch
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def train_mlp(X_tr, y_tr, X_va, y_va, input_dim, hidden_dim, num_classes):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MulticlassMLP(input_dim, hidden_dim, num_classes).to(device)

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_tr, dtype=torch.long).to(device)
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
    y_va_t = torch.tensor(y_va, dtype=torch.long).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss()
    stopper = EarlyStopping()

    print(f"  Architecture: Linear({input_dim}->{hidden_dim}) -> BN({hidden_dim}) "
          f"-> ReLU -> Linear({hidden_dim}->{num_classes})")
    print(f"  Device={device}, epochs={EPOCHS}, batch_size={BATCH_SIZE}")

    n = X_tr_t.shape[0]
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            opt.zero_grad()
            ce(model(X_tr_t[idx]), y_tr_t[idx]).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = ce(model(X_va_t), y_va_t).item()

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"    epoch {epoch+1:>4d}  val_loss={val_loss:.6f}")

        if stopper.step(val_loss, epoch + 1, model):
            print(f"  Early stopping @ epoch {epoch+1} "
                  f"(best={stopper.best_loss:.6f} @ {stopper.best_epoch})")
            stopper.restore(model)
            break
    else:
        stopper.restore(model)
    model.cpu(); model.eval()
    return model


# ============================================================
# CIFAR-10 loader (identical to full-train)
# ============================================================
CIFAR10_CLASS_NAMES = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat", 4: "deer",
    5: "dog",      6: "frog",       7: "horse", 8: "ship", 9: "truck",
}


def load_cifar10(data_root=CIFAR_DATA_ROOT):
    print(f"Loading CIFAR-10 from {data_root}/cifar-10-batches-py/ ...")
    try:
        from torchvision.datasets import CIFAR10
    except ImportError as e:
        raise ImportError("torchvision is required. Install via `pip install torchvision`.") from e

    try:
        train_ds = CIFAR10(root=data_root, train=True,  download=False)
        test_ds  = CIFAR10(root=data_root, train=False, download=False)
    except Exception as e:
        raise FileNotFoundError(
            f"Could not load CIFAR-10 from '{data_root}'.\n"
            f"Expected '{data_root}/cifar-10-batches-py/' to exist.\n"
            f"If you have 'cifar-10-python.tar.gz', extract it:\n"
            f"  cd {data_root} && tar -xzf cifar-10-python.tar.gz\n"
            f"Original error: {e}")

    X_train = train_ds.data.reshape(len(train_ds), -1).astype(np.float32) / 255.0
    y_train = np.array(train_ds.targets, dtype=np.int64)
    X_test  = test_ds.data.reshape(len(test_ds), -1).astype(np.float32) / 255.0
    y_test  = np.array(test_ds.targets, dtype=np.int64)

    print(f"  CIFAR-10 classes ({NUM_CLASSES}): {CIFAR10_CLASS_NAMES}")
    print(f"  Raw flat shape: train={X_train.shape}, test={X_test.shape}")
    print(f"  Train class counts: {np.bincount(y_train).tolist()}")
    print(f"  Test class counts:  {np.bincount(y_test).tolist()}")
    return X_train, y_train, X_test, y_test


def extract_test_metrics(result, method_name):
    if result.report_vs_truth is None:
        return None
    df = result.report_vs_truth
    row = df[(df["Method"] == method_name) & (df["Split"] == "test")]
    if len(row) == 0:
        return None
    return row.iloc[0].to_dict()


# ============================================================
# Split train.csv indices into train_net / val_net / pool_max / (sub-)pool
# ============================================================
def split_train_indices(y_train_all, total_samples, pool_frac):
    """
    Split the indices of CIFAR-10 train (50k) into:
      - train_net : remainder (~32k), used to train the MLP + fit PCA/Scaler
      - val_net   : VAL_NET_FRAC  * total_samples (~6k), for MLP early stopping
      - pool_max  : POOL_MAX_FRAC * total_samples (~12k), disjoint from val_net and train_net
      - pool      : pool_frac     * total_samples  -- subsampled from pool_max (or == pool_max if pool_frac >= POOL_MAX_FRAC)
      - rep_fit / rep_sel : pool split 50/50
    """
    n_train = len(y_train_all)
    val_net_size  = int(round(VAL_NET_FRAC  * total_samples))
    pool_max_size = int(round(POOL_MAX_FRAC * total_samples))
    assert val_net_size + pool_max_size < n_train, \
        "val_net + pool_max must be smaller than train set size"

    idx_train = np.arange(n_train)

    # First split out val_net from train (stratified)
    idx_rest, idx_val_net, y_rest, _ = train_test_split(
        idx_train, y_train_all,
        test_size=val_net_size,
        random_state=SEED, stratify=y_train_all,
    )
    # From the remainder, split out pool_max (stratified on remainder labels)
    idx_train_net, idx_pool_max, y_train_net, y_pool_max = train_test_split(
        idx_rest, y_rest,
        test_size=pool_max_size,
        random_state=SEED + 2, stratify=y_rest,
    )

    # Sub-sample pool_max to get the actual pool (if pool_frac < POOL_MAX_FRAC)
    if pool_frac >= POOL_MAX_FRAC - 1e-6:
        idx_pool, y_pool = idx_pool_max, y_pool_max
    else:
        keep = pool_frac / POOL_MAX_FRAC
        idx_pool, _, y_pool, _ = train_test_split(
            idx_pool_max, y_pool_max, train_size=keep,
            random_state=SEED + 3, stratify=y_pool_max)

    # Split pool into rep_fit / rep_sel
    idx_rep_fit, idx_rep_sel, _, _ = train_test_split(
        idx_pool, y_pool, test_size=REPLACEMENT_SELECTION_FRACTION,
        random_state=SEED + 4, stratify=y_pool)

    return {
        "train_net": idx_train_net,
        "val_net":   idx_val_net,
        "rep_fit":   idx_rep_fit,
        "rep_sel":   idx_rep_sel,
    }


# ============================================================
# Main
# ============================================================
def main():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    print("\n" + "=" * 78)
    print("Step 1: Load CIFAR-10 (flatten + normalize to [0,1])")
    print("=" * 78)
    X_train_all_raw, y_train_all, X_test_raw, y_test = load_cifar10()

    total_samples = len(y_train_all) + len(y_test)
    print(f"  Total samples across train+test: {total_samples}")
    print(f"  Pool sizes (as % of total): {[f'{p*100:.0f}%' for p in POOL_SIZES_TO_COMPARE]}")

    print("\n" + "=" * 78)
    print("Step 2: Partition train into train_net + val_net + pool_max")
    print("=" * 78)
    max_pool = max(POOL_SIZES_TO_COMPARE)
    base_splits = split_train_indices(y_train_all, total_samples, pool_frac=max_pool)
    idx_train_net = base_splits["train_net"]
    idx_val_net   = base_splits["val_net"]

    X_train_net_raw = X_train_all_raw[idx_train_net]
    y_train_net     = y_train_all[idx_train_net]
    X_val_net_raw   = X_train_all_raw[idx_val_net]
    y_val_net       = y_train_all[idx_val_net]

    n_pool_max = len(y_train_all) - len(idx_train_net) - len(idx_val_net)
    print(f"  train_net : n={len(idx_train_net):5d}   (used to train MLP + fit PCA/Scaler)")
    print(f"  val_net   : n={len(idx_val_net):5d}   (for MLP early stopping)")
    print(f"  pool_max  : n={n_pool_max:5d}   (disjoint; holds out the replacement pool)")
    print(f"  test      : n={len(y_test):5d}   (CIFAR-10 test_batch)")
    print(f"  train_net class counts: {np.bincount(y_train_net, minlength=NUM_CLASSES).tolist()}")

    print("\n" + "=" * 78)
    print("Step 2b: PCA (fit on train_net) + StandardScaler")
    print("=" * 78)
    n_comp = min(PCA_COMPONENTS, X_train_net_raw.shape[1] - 1, X_train_net_raw.shape[0] - 1)
    if n_comp < PCA_COMPONENTS:
        print(f"  Note: reducing PCA n_components from {PCA_COMPONENTS} to {n_comp}")
    print(f"  PCA: n_components={n_comp}, svd_solver='randomized'")
    pca = PCA(n_components=n_comp, svd_solver="randomized", random_state=SEED)
    Xtrn_pca = pca.fit_transform(X_train_net_raw).astype(np.float64)
    Xval_pca = pca.transform(X_val_net_raw).astype(np.float64)
    Xte_pca  = pca.transform(X_test_raw).astype(np.float64)
    print(f"  PCA explained variance: {pca.explained_variance_ratio_.sum():.4f}")

    scaler = StandardScaler().fit(Xtrn_pca)
    X_train_net = scaler.transform(Xtrn_pca).astype(np.float64)
    X_val_net   = scaler.transform(Xval_pca).astype(np.float64)
    X_test      = scaler.transform(Xte_pca).astype(np.float64)
    d = X_train_net.shape[1]
    print(f"  Final dense shape: {X_train_net.shape}")

    print("\n" + "=" * 78)
    print("Step 3: Train MLP once")
    print("=" * 78)
    model = train_mlp(X_train_net, y_train_net, X_val_net, y_val_net,
                      d, HIDDEN_DIM, NUM_CLASSES)

    with torch.no_grad():
        logits_test = model(torch.tensor(X_test, dtype=torch.float32)).cpu().numpy()
    orig_acc = float(np.mean(logits_test.argmax(axis=1) == y_test))
    print(f"\n  Original model test top-1 ACC = {orig_acc:.4f}")

    print("\n" + "=" * 78)
    print("Step 4: quad4fhe.replace(...) per pool size")
    print("=" * 78)
    all_pool_results = {}

    for pool_frac in POOL_SIZES_TO_COMPARE:
        print("\n" + "-" * 78)
        print(f"POOL = {pool_frac*100:.0f}%")
        print("-" * 78)

        splits_p = split_train_indices(y_train_all, total_samples, pool_frac=pool_frac)
        idx_rep_fit = splits_p["rep_fit"]
        y_rep_fit   = y_train_all[idx_rep_fit]

        if len(idx_rep_fit) < MIN_POOL_SAMPLES:
            print(f"  [SKIPPED] rep_fit too small (n={len(idx_rep_fit)} "
                  f"< MIN_POOL_SAMPLES={MIN_POOL_SAMPLES})")
            all_pool_results[pool_frac] = None
            _autosave_record_skipped(
                key=f"pool={int(pool_frac*100):02d}%",
                hidden_dim=HIDDEN_DIM,
                pool_fraction=pool_frac,
                rep_fit_size=None,
                reason="skipped by notebook guard (too small, missing class, or invalid stratified split)",
            )
            continue

        counts = np.bincount(y_rep_fit, minlength=NUM_CLASSES)
        if np.any(counts < MIN_SAMPLES_PER_CLASS):
            print(f"  [SKIPPED] Some class has fewer than "
                  f"{MIN_SAMPLES_PER_CLASS} samples. Class counts: {counts.tolist()}")
            all_pool_results[pool_frac] = None
            _autosave_record_skipped(
                key=f"pool={int(pool_frac*100):02d}%",
                hidden_dim=HIDDEN_DIM,
                pool_fraction=pool_frac,
                rep_fit_size=None,
                reason="skipped by notebook guard (too small, missing class, or invalid stratified split)",
            )
            continue

        # Transform rep_fit using PCA+Scaler fitted on train_net
        X_rep_fit_raw = X_train_all_raw[idx_rep_fit]
        X_rep_fit_pca = pca.transform(X_rep_fit_raw).astype(np.float64)
        X_rep_fit     = scaler.transform(X_rep_fit_pca).astype(np.float64)

        print(f"  rep_fit: n={len(X_rep_fit)}, per-class counts: {counts.tolist()}")

        result = quad4fhe.replace(
            task              = "multiclass",
            model             = model,
            hidden_layer      = "fc1",
            output_layer      = "fc2",
            activation        = "relu",
            fit_data          = (X_rep_fit, y_rep_fit),
            test_data         = (X_test, y_test),
            baselines         = BASELINES,
            export_he_dir     = f"he_artifacts_cifar10_pool_{int(pool_frac*100):02d}",
            use_cutting_plane = False,
            use_osqp_warmstart= False,
            verbose           = True,
            seed              = SEED,
        )
        all_pool_results[pool_frac] = result

        _autosave_record_result(
            result=result,
            key=f"pool={int(pool_frac*100):02d}%",
            hidden_dim=HIDDEN_DIM,
            fit_n=len(X_rep_fit),
            test_n=len(X_test),
            pool_fraction=pool_frac,
            rep_fit_size=len(X_rep_fit),
            extra={"pool": f"{int(pool_frac*100):02d}%"},
        )

        print(f"  method_used={result.method_used}, "
              f"hard_feasible={result.feasible}, "
              f"quant_decimals={result.quant_decimals}")
        q4 = extract_test_metrics(result, "Quad4FHE")
        if q4 is not None:
            acc = q4.get("ACC",       float("nan"))
            f1m = q4.get("MacroF1",   float("nan"))
            agr = q4.get("Agreement", float("nan"))
            print(f"  Quad4FHE test ACC={acc:.4f}  MacroF1={f1m:.4f}  Agreement={agr:.4f}")

    # --- Cross-pool tables ---
    print("\n" + "=" * 78)
    print("Step 5: Cross-pool comparison tables")
    print("=" * 78)

    method_order = ["Original", "Quad4FHE"] + BASELINES
    pool_labels = [f"{int(p*100):02d}%" for p in POOL_SIZES_TO_COMPARE]

    first_result = next((r for r in all_pool_results.values() if r is not None), None)
    if first_result is not None and first_result.report_vs_truth is not None:
        available_cols = list(first_result.report_vs_truth.columns)
        print(f"  Report columns: {available_cols}")
    else:
        available_cols = []

    METRIC_KEYS = [c for c in ["ACC", "MacroF1", "MacroPrec", "MacroRec", "AUC_ovr", "Agreement"]
                   if c in available_cols]

    def _build_matrix(metric_key):
        data = {}
        for pf, lbl in zip(POOL_SIZES_TO_COMPARE, pool_labels):
            result = all_pool_results[pf]
            col = {}
            for m in method_order:
                if result is None:
                    col[m] = float("nan")
                else:
                    row = extract_test_metrics(result, m)
                    col[m] = row[metric_key] if row is not None else float("nan")
            data[f"pool={lbl}"] = col
        return pd.DataFrame(data)

    for metric_key in METRIC_KEYS:
        print(f"\n--- Table: Test {metric_key} (ACC/F1/etc. vs TRUE labels; Agreement vs original model) ---")
        df = _build_matrix(metric_key)
        with pd.option_context("display.float_format", "{:.4f}".format):
            print(df.to_string())

    print("\n--- Meta Table ---")
    meta_rows = []
    for pf, lbl in zip(POOL_SIZES_TO_COMPARE, pool_labels):
        r = all_pool_results[pf]
        if r is None:
            meta_rows.append({"pool": lbl, "status": "SKIPPED"})
            continue
        meta_rows.append({
            "pool":             lbl,
            "method_used":      r.method_used,
            "hard_feasible":    r.feasible,
            "empirical_margin": round(r.empirical_margin, 6),
            "norm_margin":      round(r.normalized_margin, 6),
            "quant_decimals":   r.quant_decimals,
            "he_export_dir":    os.path.basename(r.he_export_dir) if r.he_export_dir else None,
        })
    print(pd.DataFrame(meta_rows).to_string(index=False))

    print("\n" + "=" * 78)
    print("Done.")
    print("=" * 78)


if __name__ == "__main__":
    run_notebook_main_with_autosave(main, LOG_PATH)


[autosave] stdout/stderr log -> ..\results\CIFAR10\smallpool\CIFAR10_smallpool_result.txt
[autosave] dataset=CIFAR10 experiment=smallpool
[autosave] output_dir=..\results\CIFAR10\smallpool

Step 1: Load CIFAR-10 (flatten + normalize to [0,1])
Loading CIFAR-10 from ./data/cifar-10-batches-py/ ...
  CIFAR-10 classes (10): {0: 'airplane', 1: 'automobile', 2: 'bird', 3: 'cat', 4: 'deer', 5: 'dog', 6: 'frog', 7: 'horse', 8: 'ship', 9: 'truck'}
  Raw flat shape: train=(50000, 3072), test=(10000, 3072)
  Train class counts: [5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]
  Test class counts:  [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
  Total samples across train+test: 60000
  Pool sizes (as % of total): ['1%', '5%', '10%', '20%']

Step 2: Partition train into train_net + val_net + pool_max
  train_net : n=32000   (used to train MLP + fit PCA/Scaler)
  val_net   : n= 6000   (for MLP early stopping)
  pool_max  : n=12000   (disjoint; holds out the replacement poo